In [2]:
import os
os.chdir("..")

from utils.utils import notebook_line_magic
notebook_line_magic()

Line Magic Set


## CQL

In [3]:
import warnings
warnings.simplefilter('ignore')

from utils.utils import set_ld_library_path
from ding.entry import serial_pipeline_offline
from dizoo.d4rl.config.halfcheetah_random_cql_config import main_config, create_config
from ding.config import compile_config

set_ld_library_path()

[09-08 17:45:19] WARNING  If you want to use numba to speed up segment tree, please install numba first                                              ]8;id=813590;file:///home/azm0269@auburn.edu/.pyenv/versions/3.10.18/lib/python3.10/site-packages/ding/utils/default_helper.py\default_helper.py]8;;\:]8;id=396194;file:///home/azm0269@auburn.edu/.pyenv/versions/3.10.18/lib/python3.10/site-packages/ding/utils/default_helper.py#450\450]8;;\

MUJOCO_GL = osmesa
MUJOCO_PY_MUJOCO_PATH = /home/azm0269@auburn.edu/.mujoco/mujoco210
LD_LIBRARY_PATH entries:
   /home/azm0269@auburn.edu/.mujoco/mujoco210/bin
   /home/azm0269@auburn.edu/.pyenv/versions/3.10.18/lib/python3.10/site-packages/mujoco_py/generated/_pyxbld_2.1.2.14_310_linuxcpuextensionbuilder/lib.linux-x86_64-cpython-310/mujoco_py


In [4]:
main_config.exp_name = "generated_artifacts/halfcheetah_expert_cql_seed0"

In [5]:
cfg = compile_config(
    main_config,
    create_cfg=create_config,
    auto=True,
)

These new versions include large bug fixes, new versions of Python, and are where all new development will continue. Please upgrade these libraries as soon as you're able to do so.
If you'd like to read more about the story behind this switch, please check out ]8;;https://farama.org/Announcing-Minari\this blog post]8;;\.
<frozen importlib._bootstrap>:283: DeprecationWarning: the load_module() method is deprecated and slated for removal in Python 3.12; use exec_module() instead
No module named 'flow'
No module named 'carla'
pybullet build time: Jan 29 2025 23:16:28


In [29]:
# cfg.policy.model

In [6]:
import gym
from dizoo.d4rl.envs.d4rl_env import D4RLEnv
from ding.envs import DingEnvWrapper, BaseEnvManagerV2

# collector_env = BaseEnvManagerV2(
#     env_fn=[lambda: DingEnvWrapper(gym.make("HalfCheetah-v2")) for _ in range(cfg.env.collector_env_num)],
#     cfg=cfg.env.manager
# )
# evaluator_env = BaseEnvManagerV2(
#     env_fn=[lambda: DingEnvWrapper(gym.make("HalfCheetah-v2")) for _ in range(cfg.env.evaluator_env_num)],
#     cfg=cfg.env.manager
# )
# cfg.env

In [7]:
# serial_pipeline_offline([main_config, create_config], seed=0)
from ding.model import ContinuousQAC
from ding.policy import CQLPolicy
from ding.data import DequeBuffer

model = ContinuousQAC(**cfg.policy.model)
buffer_ = DequeBuffer(size=cfg.policy.other.replay_buffer.replay_buffer_size)
policy = CQLPolicy(cfg=cfg.policy, model=model)

In [8]:
# import 

In [9]:
# from d4rl.locomotion import ant
from ding.utils import set_pkg_seed
from ding.framework import task, ding_init
from ding.data import create_dataset
from ding.framework.context import OfflineRLContext
from ding.framework.middleware import interaction_evaluator, trainer, CkptSaver, offline_data_fetcher, offline_logger

ding_init(cfg)
with task.start(async_mode=False, ctx=OfflineRLContext()):
    # Evaluating, we place it on the first place to get the score of the random model as a benchmark value
    evaluator_env = BaseEnvManagerV2(
        env_fn=[lambda: D4RLEnv(cfg.env) for _ in range(cfg.env.evaluator_env_num)], cfg=cfg.env.manager
    )
    set_pkg_seed(cfg.seed, use_cuda=cfg.policy.cuda)
    
    dataset = create_dataset(cfg)
    model = ContinuousQAC(**cfg.policy.model)
    policy = CQLPolicy(cfg.policy, model=model)
    
    task.use(interaction_evaluator(cfg, policy.eval_mode, evaluator_env))
    task.use(offline_data_fetcher(cfg, dataset))
    task.use(trainer(cfg, policy.learn_mode))
    task.use(CkptSaver(policy, cfg.exp_name, train_freq=100))
    task.use(offline_logger())
    task.run()

/home/azm0269@auburn.edu/.pyenv/versions/3.10.18/lib/python3.10/site-packages/gym/core.py:329: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/home/azm0269@auburn.edu/.pyenv/versions/3.10.18/lib/python3.10/site-packages/gym/wrappers/step_api_compatibility.py:39: DeprecationWarning: WARN: Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
load datafile: 100%|██████████| 21/21 [00:01<00:00, 12.31it/s]
/home/azm0269@auburn.edu/.pyenv/versions/3.10.18/lib/python3.10/site-packages/gym/core.py:268: DeprecationWarning: WARN: Function `env.seed(seed)` is marked as deprecated and will be removed in the future. Please use `env.reset(seed=seed)` instead.
  deprecat

KeyboardInterrupt: 

In [8]:
cfg

{'env': {'manager': {'episode_num': inf,
   'max_retry': 1,
   'retry_type': 'reset',
   'auto_reset': True,
   'step_timeout': None,
   'reset_timeout': None,
   'retry_waiting_time': 0.1,
   'cfg_type': 'BaseEnvManagerDict',
   'type': 'base'},
  'stop_value': 6000,
  'n_evaluator_episode': 8,
  'type': 'd4rl',
  'import_names': ['dizoo.d4rl.envs.d4rl_env'],
  'env_id': 'halfcheetah-expert-v2',
  'collector_env_num': 1,
  'evaluator_env_num': 8,
  'use_act_scale': True},
 'policy': {'model': {'twin_critic': True,
   'action_space': 'reparameterization',
   'actor_head_hidden_size': 256,
   'critic_head_hidden_size': 256,
   'obs_shape': 17,
   'action_shape': 6},
  'learn': {'learner': {'train_iterations': 1000000000,
    'dataloader': {'num_workers': 0},
    'log_policy': True,
    'hook': {'load_ckpt_before_run': '',
     'log_show_after_iter': 100,
     'save_ckpt_after_iter': 10000,
     'save_ckpt_after_run': True},
    'cfg_type': 'BaseLearnerDict'},
   'resume_training': False